# Deep Learning Data Preprocessing Pipeline (BTCUSDT)

This notebook builds a production-ready preprocessing pipeline for:
- 15m, 1h, 4h, 1d, 1w, 1mo processed tables

It performs:
1. Data cleaning and column filtering
2. Missing value handling by policy
3. Feature engineering
4. Multi-timeframe alignment (15m base + latest 1h/4h)
5. CSV export to data/processed

In [1]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.preprocess_for_deep_learning import (
    TABLES,
    KEEP_FEATURES,
    DROP_COLUMNS,
    run_pipeline,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## Configuration Summary

In [2]:
print("Input tables:")
for tf, table in TABLES.items():
    print(f"- {tf}: {table}")

print("\nDropped columns (explicit):")
for c in sorted(DROP_COLUMNS):
    print(f"- {c}")

print("\nKept core features:")
for c in KEEP_FEATURES:
    print(f"- {c}")

Input tables:
- 15m: btcusdt_15m_processed
- 1h: btcusdt_1h_processed
- 4h: btcusdt_4h_processed
- 1d: btcusdt_1d_processed
- 1w: btcusdt_1w_processed
- 1mo: btcusdt_1mo_processed

Dropped columns (explicit):
- candle_pattern_1
- candle_pattern_2
- candle_pattern_3
- candle_pattern_complex
- chart_pattern_key_points

Kept core features:
- open
- high
- low
- close
- volume
- rsi_14
- macd
- macd_signal
- macd_hist
- ema_12
- ema_26
- atr_14
- adx_14
- mfi_14
- trend_pct_change
- vix
- volume_oscillator
- support
- resistance


## Run Pipeline and Export CSVs

In [3]:
output_dir = project_root / "data" / "processed"
exported = run_pipeline(output_dir=output_dir)

print("Export completed:")
for name, path in exported.items():
    print(f"- {name}: {path}")

Export completed:
- cleaned_15m: /Users/kisho/Projects/PeekNode/crypto_data/crypto_data_platform/data/processed/cleaned_15m.csv
- cleaned_1h: /Users/kisho/Projects/PeekNode/crypto_data/crypto_data_platform/data/processed/cleaned_1h.csv
- cleaned_4h: /Users/kisho/Projects/PeekNode/crypto_data/crypto_data_platform/data/processed/cleaned_4h.csv
- merged_multi_timeframe: /Users/kisho/Projects/PeekNode/crypto_data/crypto_data_platform/data/processed/merged_multi_timeframe.csv


## Validate Output Files and Missing Values

In [4]:
summary_rows = []
for name, path in exported.items():
    df = pd.read_csv(path)
    non_numeric_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    summary_rows.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "total_missing": int(df.isna().sum().sum()),
        "non_numeric_columns": len(non_numeric_cols),
        "is_sorted_open_time_ms": bool(df["open_time_ms"].is_monotonic_increasing) if "open_time_ms" in df.columns else False,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("dataset").reset_index(drop=True)
summary_df

,dataset,rows,columns,total_missing,non_numeric_columns,is_sorted_open_time_ms
0,cleaned_15m,303071,23,0,0,True
1,cleaned_1h,75768,23,0,0,True
2,cleaned_4h,18942,23,0,0,True
3,merged_multi_timeframe,303071,67,0,0,True


## Example Output CSV Format (Columns)

In [5]:
for name, path in exported.items():
    df = pd.read_csv(path, nrows=1)
    print(f"\n{name} -> {path.name}")
    print(df.columns.tolist())


cleaned_15m -> cleaned_15m.csv
['open_time_ms', 'open', 'high', 'low', 'close', 'volume', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'ema_12', 'ema_26', 'atr_14', 'adx_14', 'mfi_14', 'trend_pct_change', 'vix', 'volume_oscillator', 'support', 'resistance', 'return', 'high_low_range', 'body_size']

cleaned_1h -> cleaned_1h.csv
['open_time_ms', 'open', 'high', 'low', 'close', 'volume', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'ema_12', 'ema_26', 'atr_14', 'adx_14', 'mfi_14', 'trend_pct_change', 'vix', 'volume_oscillator', 'support', 'resistance', 'return', 'high_low_range', 'body_size']

cleaned_4h -> cleaned_4h.csv
['open_time_ms', 'open', 'high', 'low', 'close', 'volume', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'ema_12', 'ema_26', 'atr_14', 'adx_14', 'mfi_14', 'trend_pct_change', 'vix', 'volume_oscillator', 'support', 'resistance', 'return', 'high_low_range', 'body_size']

merged_multi_timeframe -> merged_multi_timeframe.csv
['open_time_ms', 'open', 'high', 'low', 'close'

## Expected Folder Structure

```
crypto_data_platform/
  scripts/
    preprocess_for_deep_learning.py
  data/
    processed/
      cleaned_15m.csv
      cleaned_1h.csv
      cleaned_4h.csv
      merged_multi_timeframe.csv
```